# UPC
## CC219 — Aplicaciones de Data Science
### Trabajo parcial — Hito 1 · 2026-02

# 01. Carga e inspección inicial de MovieLens

**Proyecto:** Sistema de recomendación de películas con MovieLens.

**Responsables de la sección:** Piero Marcos Contreras Albornoz.

## Objetivo
Cargar los cuatro CSV, documentar sus variables, comprobar su calidad y medir la cobertura inicial del catálogo. Este notebook realiza inspección: las decisiones de limpieza se desarrollarán en el notebook 03.

## Mapa del notebook
1. Definición del problema NLP.
2. Procedencia y descripción de los datos.
3. Preparación del entorno y verificación de archivos.
4. Carga e inspección de los cuatro CSV.
5. Diccionario de variables.
6. Inspección de calidad.
7. Relaciones y cobertura inicial.
8. Conclusiones y transición al EDA.

**Reproducibilidad:** ejecutar las celdas en orden con Python, pandas y NumPy instalados. Las rutas se resuelven desde la raíz del proyecto o desde `code/`. No se sobrescriben archivos originales ni se entrenan modelos.

# 1. Definición del problema NLP

## 1.1. Problema y caso de uso
El proyecto aborda el descubrimiento de películas acordes con los intereses de un usuario. El caso de uso plantea apoyar esa elección mediante recomendaciones que combinen el historial de valoraciones con el contenido textual asociado a las películas. La utilidad de la propuesta se evaluará en una etapa posterior; este parcial establece la disponibilidad y calidad de los datos necesarios.

## 1.2. Componente de lenguaje natural
El texto no estructurado procede de **`tags.csv`**: palabras o frases libres asignadas por usuarios. Distintas personas pueden describir la misma película de maneras diferentes, con variaciones de escritura y vocabulario. Por ello, el análisis NLP requiere examinar la cobertura, normalizar con cuidado y agrupar las etiquetas por película. MovieLens Latest Small no contiene sinopsis.

La propuesta del README contempla **TF-IDF** para representar ese texto y **similitud coseno** para comparar películas y perfiles de preferencias. También contempla **regresión logística multietiqueta** para identificar géneros a partir del texto. En clasificación, los géneros serán objetivos y no formarán parte del texto de entrada.

## 1.3. Preguntas del proyecto

| Tarea | Pregunta | Información que se utilizaría | Evaluación propuesta |
| --- | --- | --- | --- |
| Clasificación multietiqueta | ¿Qué géneros podemos identificar en una película a partir de las etiquetas textuales asignadas por los usuarios? | Etiquetas como entrada; géneros como objetivos. | F1 micro, macro y por género. |
| Predicción de relevancia | ¿Qué películas tendrán mayor afinidad con un usuario a partir de sus valoraciones anteriores y del contenido textual de las películas que prefiere? | Historial favorable y texto de películas; candidatos no vistos. | Precision@K y Recall@K. |

## 1.4. Alcance del Hito 1
Este notebook carga, describe e inspecciona los datos. El notebook 02 desarrolla el EDA y el 03 prepara los datos. La propuesta de modelización explicará el criterio de valoración favorable, la separación de datos y la prevención de información futura. El entrenamiento, el cálculo de métricas de modelos y la implementación del recomendador quedan fuera de esta entrega.

La definición del problema y las preguntas siguen el **README del proyecto**. La cobertura textual observada delimitará qué parte del catálogo puede participar en la propuesta basada en etiquetas.

# 2. Procedencia y descripción de los datos

## 2.1. Ficha de procedencia

| Campo | Descripción |
| --- | --- |
| Fuente | GroupLens, Universidad de Minnesota; interacciones del servicio MovieLens. |
| Dataset | MovieLens Latest Small (`ml-latest-small`). |
| Edición descrita en el README | Generada el 26 de septiembre de 2018. |
| Archivos locales | `data/raw/movies.csv`, `ratings.csv`, `tags.csv` y `links.csv`. |
| Tamaño de referencia | 9 742 películas, 100 836 valoraciones, 3 683 asignaciones de etiquetas y 610 usuarios. Los conteos locales se comprobarán en la carga. |
| Período de referencia | Del 29 de marzo de 1996 al 24 de septiembre de 2018. |
| Formato | CSV con encabezado, separador coma y codificación UTF-8. |
| Texto para NLP | Etiquetas libres de usuarios; no incluye sinopsis. |
| Usuarios | Identificadores anónimos, sin perfiles demográficos. |
| Valoraciones | De 0,5 a 5 estrellas, en pasos de 0,5. |
| Fechas | Timestamps Unix en segundos; interpretación en UTC. |
| Descarga | [MovieLens Latest Datasets](https://grouplens.org/datasets/movielens/latest/). |
| Documentación original | [Descripción y condiciones de uso de GroupLens](https://files.grouplens.org/datasets/movielens/ml-latest-small-README.html). |
| Fecha de descarga local | No documentada; no se infiere de las fechas de modificación. |

## 2.2. Contenido y significado de las columnas

### movies.csv: catálogo de películas
Contiene 9 742 filas, una por película, y tres columnas.

| Columna | Qué representa | Uso e interpretación |
| --- | --- | --- |
| `movieId` | Identificador interno de la película en MovieLens. | Clave para relacionar los cuatro archivos; no mide popularidad ni orden de estreno. |
| `title` | Título registrado, que puede incluir el año entre paréntesis. | Permite identificar películas e interpretar ejemplos; no es una sinopsis ni reemplaza las etiquetas ausentes. |
| `genres` | Uno o varios géneros separados por `&#124;`. | Objetivos de la clasificación multietiqueta. `(no genres listed)` indica género no informado. |

Una película puede pertenecer a varios géneros. Para clasificación, estos géneros deben mantenerse separados del texto de entrada: incorporarlos revelaría la respuesta que se intenta predecir.

### ratings.csv: valoraciones de usuarios
Contiene 100 836 filas, una por evento de valoración, y cuatro columnas. El número de eventos no es el número de usuarios o películas únicos.

| Columna | Qué representa | Uso e interpretación |
| --- | --- | --- |
| `userId` | Usuario anónimo que valoró la película. | Agrupa su historial y comparte identificadores con tags; no contiene datos demográficos. |
| `movieId` | Película valorada. | Relaciona la preferencia con el catálogo y su texto disponible. |
| `rating` | Puntuación de 0,5 a 5, en pasos de 0,5. | Señal explícita de preferencia; se definirá qué valor se considera favorable. |
| `timestamp` | Momento de la valoración en segundos Unix UTC. | Ordena el historial y permite diseñar cortes temporales para evitar información futura. |

Las valoraciones complementan el componente NLP con preferencias observadas. Una película sin valoración no tiene una puntuación de cero ni necesariamente fue rechazada por el usuario.

### tags.csv: fuente textual para NLP
Contiene 3 683 filas, una por asignación de etiqueta, y cuatro columnas. Una película puede recibir varias etiquetas y el mismo texto puede aparecer en eventos distintos.

| Columna | Qué representa | Uso e interpretación |
| --- | --- | --- |
| `userId` | Usuario que asignó la etiqueta. | Conserva la procedencia y permite estudiar quién aporta el texto. |
| `movieId` | Película descrita por la etiqueta. | Permite agrupar etiquetas en un documento por película. |
| `tag` | Palabra o frase libre escrita por el usuario. | Entrada no estructurada del proyecto: puede describir temas, nombres u otros aspectos; no equivale a una sinopsis ni a un género oficial. |
| `timestamp` | Momento de asignación en segundos Unix UTC. | Permite usar solo texto disponible hasta el corte temporal elegido. |

El número de filas mide asignaciones, no etiquetas únicas ni películas con texto. La normalización y el tratamiento de repeticiones se documentarán en el notebook 03. Las etiquetas reflejan descripciones de sus autores, sin garantizar una caracterización completa de las películas.

### links.csv: referencias externas de películas
Contiene 9 742 filas y tres columnas, que relacionan películas con identificadores externos.

| Columna | Qué representa | Uso e interpretación |
| --- | --- | --- |
| `movieId` | Identificador interno de MovieLens. | Relaciona cada referencia con el catálogo; se comprueba unicidad e integridad. |
| `imdbId` | Identificador de IMDb sin el prefijo `tt`. | Se conserva como texto para mantener ceros iniciales, por ejemplo `0114709`; no es una cantidad numérica. |
| `tmdbId` | Identificador de The Movie Database (TMDB). | Entero anulable porque algunas referencias faltan; un nulo no implica que falte la película en MovieLens. |

Links no aporta etiquetas, sinopsis ni valoraciones. Tener referencias externas no significa que se haya descargado información de esos servicios; aquí solo se inspecciona su correspondencia con el catálogo.

## 2.3. Atribución y condiciones de uso
Reconocer el uso de MovieLens y conservar las condiciones originales al redistribuir el dataset o sus transformaciones. La documentación de GroupLens establece requisitos adicionales para usos comerciales; consultar la fuente enlazada para las condiciones completas.

Referencia: Harper, F. M. y Konstan, J. A. (2015). *The MovieLens Datasets: History and Context*. [DOI](https://doi.org/10.1145/2827872).

# 3. Preparación del entorno y verificación de archivos

Se resuelven rutas relativas desde la raíz o desde `code/`, se verifican los cuatro CSV y se calculan huellas SHA-256. Estas huellas permiten comprobar al final que la inspección no modificó los originales. Los datos limpios se producirán posteriormente en `data/processed/`.

In [ ]:
from pathlib import Path
import hashlib
import platform
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

In [ ]:
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 90)

def mostrar_tabla(titulo, tabla):
    """Mostrar un título junto a cada tabla para facilitar su lectura."""
    display(Markdown(f"#### {titulo}"))
    display(tabla)

candidatos = [Path.cwd(), Path.cwd().parent]

RAIZ = next((p for p in candidatos if (p / "data" / "raw" / "movies.csv").is_file()), None)

if RAIZ is None:
    raise FileNotFoundError("Ejecutar desde la raíz del proyecto o desde code/ con los CSV en data/raw/.")
RAW = RAIZ / "data" / "raw"
ARCHIVOS = ["movies.csv", "ratings.csv", "tags.csv", "links.csv"]

faltantes = [nombre for nombre in ARCHIVOS if not (RAW / nombre).is_file()]

if faltantes:
    raise FileNotFoundError(f"Archivos requeridos ausentes: {faltantes}")

def huella(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

huellas_iniciales = {nombre: huella(RAW / nombre) for nombre in ARCHIVOS}
inventario = pd.DataFrame([
    {"archivo": nombre, "bytes": (RAW / nombre).stat().st_size,
     "sha256": huellas_iniciales[nombre]} for nombre in ARCHIVOS
])

mostrar_tabla('Inventario de archivos originales y huellas de integridad', inventario)

#### Inventario de archivos originales y huellas de integridad

,archivo,bytes,sha256
0,movies.csv,494431,5a5f32dd9bb3797b8e728a1b98958789d2b13f294a69fdfbc5727f8a9611aa07
1,ratings.csv,2483723,aa289ca83157595d0df6aea1be6a4ded676ddc4385472e8313a8ed9805352646
2,tags.csv,118660,92a9f8bb7916dceef6151209845788c3643f794dfa79d1feaec7121b5960399d
3,links.csv,197979,97ad18e4e56a09363c65676b6cb3482ce3e2cea2372a24620c1599c843325f31


### Interpretación: Inventario y huellas de integridad

Los cuatro CSV requeridos están disponibles. `bytes` mide el tamaño en disco, no la cantidad de observaciones. La huella SHA-256 identifica el contenido leído y permite comparar el archivo antes y después de la ejecución. Sin una huella oficial de referencia, no prueba autenticidad ni identidad con la descarga publicada por GroupLens.

In [2]:
print(f"Python {platform.python_version()} | pandas {pd.__version__} | NumPy {np.__version__}")
print("README.txt original:", "disponible" if (RAW / "README.txt").is_file() else "pendiente de incorporar en data/raw/")

Python 3.12.7 | pandas 2.2.3 | NumPy 2.3.4
README.txt original: pendiente de incorporar en data/raw/


### Interpretación: Entorno y documentación

Las versiones identifican el entorno utilizado para generar los resultados. El mensaje sobre README.txt informa si se conserva localmente la documentación original; en esta ejecución está pendiente. Esta ausencia documental no impide cargar los CSV, pero debe resolverse para conservar procedencia y condiciones de uso junto a los datos.

# 4. Carga e inspección de los cuatro archivos

Primero se registran los tipos inferidos al leer cada CSV. Después se carga una representación explícita: identificadores internos y timestamps como enteros anulables, texto como cadenas y valoraciones como números decimales. `imdbId` se conserva como cadena para no perder ceros iniciales; `tmdbId` usa un entero anulable porque puede faltar. Declarar tipos no equivale a corregir registros: un valor incompatible producirá un error visible.

In [3]:
inferidos = {nombre: pd.read_csv(RAW / nombre, encoding="utf-8") for nombre in ARCHIVOS}
TIPOS = {
    "movies.csv": {"movieId": "Int64", "title": "string", "genres": "string"},
    "ratings.csv": {"userId": "Int64", "movieId": "Int64", "rating": "Float64", "timestamp": "Int64"},
    "tags.csv": {"userId": "Int64", "movieId": "Int64", "tag": "string", "timestamp": "Int64"},
    "links.csv": {"movieId": "Int64", "imdbId": "string", "tmdbId": "Int64"},
}
tablas = {nombre: pd.read_csv(RAW / nombre, encoding="utf-8", dtype=TIPOS[nombre]) for nombre in ARCHIVOS}
movies, ratings, tags, links = [tablas[nombre] for nombre in ARCHIVOS]
resumen = pd.DataFrame([
    {"archivo": nombre, "filas": len(df), "columnas": df.shape[1],
     "memoria_MiB": round(df.memory_usage(deep=True).sum() / 1024**2, 3)}
    for nombre, df in tablas.items()
])
mostrar_tabla('Dimensiones y memoria de los cuatro CSV', resumen)

#### Dimensiones y memoria de los cuatro CSV

,archivo,filas,columnas,memoria_MiB
0,movies.csv,9742,3,1.387
1,ratings.csv,100836,4,3.462
2,tags.csv,3683,4,0.303
3,links.csv,9742,3,0.688


### Interpretación: Dimensiones y memoria

Las filas representan unidades diferentes: 9 742 películas, 100 836 valoraciones, 3 683 asignaciones de etiquetas y 9 742 correspondencias externas. No deben sumarse para obtener películas o usuarios únicos. Movies y links tienen tres columnas; ratings y tags, cuatro. La memoria en MiB estima el coste de cada tabla cargada, incluyendo cadenas e índice; puede diferir del tamaño del CSV y cambiar según el entorno, sin indicar diferencias de calidad.

### Tipos de datos y esquema de carga

In [4]:
tipos_observados = pd.DataFrame([
    {"archivo": nombre, "variable": col, "tipo_inferido": str(inferidos[nombre][col].dtype),
     "tipo_cargado": str(df[col].dtype)}
    for nombre, df in tablas.items() for col in df.columns
])
mostrar_tabla('Tipos inferidos y tipos usados en la carga', tipos_observados)

#### Tipos inferidos y tipos usados en la carga

,archivo,variable,tipo_inferido,tipo_cargado
0,movies.csv,movieId,int64,Int64
1,movies.csv,title,object,string
2,movies.csv,genres,object,string
3,ratings.csv,userId,int64,Int64
4,ratings.csv,movieId,int64,Int64
5,ratings.csv,rating,float64,Float64
6,ratings.csv,timestamp,int64,Int64
7,tags.csv,userId,int64,Int64
8,tags.csv,movieId,int64,Int64
9,tags.csv,tag,object,string


### Interpretación: Tipos inferidos y tipos cargados

Los tipos inferidos son la decisión automática de pandas; los cargados corresponden al esquema explícito. `Int64` y `Float64` permiten faltantes, pero usarlos no significa que todas las columnas tengan nulos. IMDb se lee desde el CSV como texto para conservar ceros iniciales. TMDB se carga como entero anulable porque contiene ausencias. Los identificadores no son magnitudes que deban promediarse; rating sí expresa una puntuación.

### Muestras de los cuatro archivos

In [5]:
for nombre, df in tablas.items():
    print(f"\nMuestra: {nombre}")
    mostrar_tabla(f"Primeros registros de {nombre}", df.head(3))

referencia = {"movies.csv": 9742, "ratings.csv": 100836, "tags.csv": 3683, "links.csv": 9742}
comparacion = resumen[["archivo", "filas"]].copy()
comparacion["filas_referencia"] = comparacion["archivo"].map(referencia)
comparacion["diferencia"] = comparacion["filas"] - comparacion["filas_referencia"]
mostrar_tabla('Comparación de filas locales con la edición de referencia', comparacion)


Muestra: movies.csv


#### Primeros registros de movies.csv

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance



Muestra: ratings.csv


#### Primeros registros de ratings.csv

...


Muestra: links.csv


#### Primeros registros de links.csv

,movieId,imdbId,tmdbId
0,1,0114709,862
1,2,0113497,8844
2,3,0113228,15602


#### Comparación de filas locales con la edición de referencia

,archivo,filas,filas_referencia,diferencia
0,movies.csv,9742,9742,0
1,ratings.csv,100836,100836,0
2,tags.csv,3683,3683,0
3,links.csv,9742,9742,0


### Interpretación: Muestras y diferencias de filas iguales a cero

Las primeras tres filas permiten comprobar la separación de columnas y la lectura de texto e identificadores. No constituyen una muestra aleatoria ni describen la distribución completa.

En la comparación, **diferencia = filas locales − filas de referencia**. Movies tiene 9 742 frente a 9 742; ratings, 100 836 frente a 100 836; tags, 3 683 frente a 3 683; links, 9 742 frente a 9 742. En los cuatro casos la resta da **0**: no se detecta una discrepancia en la cantidad de filas respecto de la edición descrita en el README.

Una diferencia positiva indicaría filas adicionales; una negativa, menos filas que las previstas. Cero no significa ausencia de errores: un cambio de contenido o una sustitución de registros podría conservar el total. Tampoco comprueba unicidad, faltantes o integridad de las relaciones. Por eso el conteo se complementa con controles de calidad y claves.

### Interpretación de la carga
Los cuatro archivos coinciden en cantidad de filas con la edición documentada. Esta coincidencia respalda la identificación del dataset, aunque no demuestra por sí sola que su contenido sea idéntico al original publicado. Los tipos inferidos de `imdbId` y `tmdbId` no son los más adecuados para conservar identificadores: se utiliza texto para IMDb y entero anulable para TMDB.

`links.csv` contiene identificadores externos, no sinopsis ni nuevas interacciones. Su carga permite inspeccionar su calidad y verificar su relación con el catálogo.

# 5. Diccionario de variables

El diccionario abarca **los cuatro archivos y sus 14 columnas**. Los identificadores son claves, no magnitudes numéricas para calcular promedios. Los géneros son categorías múltiples separadas por `|`; `(no genres listed)` es un indicador de información no disponible. Los títulos pueden incluir un año, pero no se asume que todos lo tengan ni se extrae todavía.

Los identificadores `userId` son consistentes entre ratings y tags. No existe un archivo de perfiles demográficos. El identificador de IMDb debe conservar sus ceros iniciales; el prefijo `tt` pertenece a las referencias externas y no está almacenado en el CSV.

In [6]:
definiciones = [
    ("movies.csv", "movieId", "Identificador interno de película", "Obligatorio, positivo y único en el catálogo"),
    ("movies.csv", "title", "Título de la película; puede incluir año", "Texto no vacío; no es una clave única"),
    ("movies.csv", "genres", "Lista de géneros separada por |", "Multietiqueta; (no genres listed) indica género no informado"),
    ("ratings.csv", "userId", "Identificador anónimo de usuario", "Obligatorio y positivo; compartido con tags"),
    ("ratings.csv", "movieId", "Película valorada", "Debe existir en movies.movieId"),
    ("ratings.csv", "rating", "Valoración en estrellas", "Entre 0,5 y 5,0; incrementos de 0,5"),
    ("ratings.csv", "timestamp", "Fecha del evento en segundos Unix", "Entero; interpretar en UTC y conservar el original"),
    ("tags.csv", "userId", "Usuario que asignó la etiqueta", "Obligatorio y positivo; compartido con ratings"),
    ("tags.csv", "movieId", "Película etiquetada", "Debe existir en movies.movieId"),
    ("tags.csv", "tag", "Palabra o frase libre escrita por un usuario", "Texto no vacío; puede repetirse entre eventos"),
    ("tags.csv", "timestamp", "Fecha de asignación en segundos Unix", "Entero; interpretar en UTC y conservar el original"),
    ("links.csv", "movieId", "Película asociada con fuentes externas", "Obligatorio, positivo, único; debe existir en movies"),
    ("links.csv", "imdbId", "Identificador externo de IMDb, sin prefijo tt", "Cadena de dígitos; conservar ceros iniciales"),
    ("links.csv", "tmdbId", "Identificador externo de TMDB", "Entero positivo si está presente; admite faltantes"),
]
diccionario = pd.DataFrame(definiciones, columns=["archivo", "variable", "significado", "restriccion"])
diccionario["tipo_cargado"] = [str(tablas[a][v].dtype) for a, v, _, _ in definiciones]
assert len(diccionario) == sum(df.shape[1] for df in tablas.values())
mostrar_tabla('Diccionario completo de variables: cuatro archivos y 14 columnas', diccionario)

#### Diccionario completo de variables: cuatro archivos y 14 columnas

,archivo,variable,significado,restriccion,tipo_cargado
0,movies.csv,movieId,Identificador interno de película,"Obligatorio, positivo y único en el catálogo",Int64
1,movies.csv,title,Título de la película; puede incluir año,Texto no vacío; no es una clave única,string
2,movies.csv,genres,Lista de géneros separada por |,Multietiqueta; (no genres listed) indica género no informado,string
3,ratings.csv,userId,Identificador anónimo de usuario,Obligatorio y positivo; compartido con tags,Int64
4,ratings.csv,movieId,Película valorada,Debe existir en movies.movieId,Int64
5,ratings.csv,rating,Valoración en estrellas,"Entre 0,5 y 5,0; incrementos de 0,5",Float64
6,ratings.csv,timestamp,Fecha del evento en segundos Unix,Entero; interpretar en UTC y conservar el original,Int64
7,tags.csv,userId,Usuario que asignó la etiqueta,Obligatorio y positivo; compartido con ratings,Int64
8,tags.csv,movieId,Película etiquetada,Debe existir en movies.movieId,Int64
9,tags.csv,tag,Palabra o frase libre escrita por un usuario,Texto no vacío; puede repetirse entre eventos,string


### Interpretación del diccionario

Las 14 columnas se documentan con significado, restricciones y tipo cargado. MovieId conserva el mismo significado entre archivos, pero es clave del catálogo en movies y referencia a este en los eventos. Tag será la entrada textual, genres el objetivo de clasificación y rating la señal de preferencia. Las restricciones indican propiedades que se deben comprobar; no sustituyen los resultados de validación.

# 6. Inspección de calidad

Se distinguen nulos, textos vacíos y espacios redundantes, duplicados exactos y repeticiones en claves de análisis. Los espacios redundantes se señalan como candidatos a normalización, sin modificar el texto. Una repetición entre películas del mismo texto de etiqueta no es por sí misma un error.

In [7]:
calidad_columnas = []
for nombre, df in tablas.items():
    for col in df.columns:
        s = df[col]
        textual = isinstance(s.dtype, pd.StringDtype)
        calidad_columnas.append({
            "archivo": nombre, "variable": col, "nulos": int(s.isna().sum()),
            "vacios_o_espacios": int(s.str.strip().eq("").fillna(False).sum()) if textual else 0,
            "espacios_redundantes": int((s.ne(s.str.strip()) | s.str.contains(r"\s{2,}", regex=True)).fillna(False).sum()) if textual else 0,
        })
mostrar_tabla('Nulos, textos vacíos y espacios redundantes por columna', pd.DataFrame(calidad_columnas))

#### Nulos, textos vacíos y espacios redundantes por columna

,archivo,variable,nulos,vacios_o_espacios,espacios_redundantes
0,movies.csv,movieId,0,0,0
1,movies.csv,title,0,0,15
2,movies.csv,genres,0,0,0
3,ratings.csv,userId,0,0,0
4,ratings.csv,movieId,0,0,0
5,ratings.csv,rating,0,0,0
6,ratings.csv,timestamp,0,0,0
7,tags.csv,userId,0,0,0
8,tags.csv,movieId,0,0,0
9,tags.csv,tag,0,0,0


### Interpretación: Nulos y calidad de cadenas

Los únicos nulos de CSV son ocho tmdbId. No se detectan cadenas vacías o formadas solo por espacios. Sí hay **15 títulos con espacios redundantes**, según la regla de espacios exteriores o secuencias de dos o más espacios: se revisarán sin eliminar películas automáticamente. No encontrar este patrón en tags no descarta variantes de mayúsculas, puntuación o vocabulario. Las 34 cadenas `(no genres listed)` no son nulos técnicos, pero representan géneros no informados y se identifican en otro control.

### Revisión de duplicados exactos

In [8]:
duplicados = pd.DataFrame([
    {"archivo": nombre, "duplicados_exactos_extra": int(df.duplicated().sum())}
    for nombre, df in tablas.items()
])
mostrar_tabla('Duplicados exactos por archivo', duplicados)

#### Duplicados exactos por archivo

,archivo,duplicados_exactos_extra
0,movies.csv,0
1,ratings.csv,0
2,tags.csv,0
3,links.csv,0


### Interpretación: Duplicados exactos iguales a cero

La columna cuenta filas completamente iguales a otra anterior, considerando todas las variables del archivo. Cero en los cuatro archivos indica que esta regla no detecta registros redundantes para eliminar. No significa que una película, un usuario o una etiqueta aparezcan una sola vez: repetir entidades entre distintos eventos puede ser legítimo. Las combinaciones de claves se revisan por separado.

### Validación de claves, puntuaciones y fechas

In [9]:
controles = []
def registrar(archivo, control, mascara, tratamiento):
    df = tablas[archivo]
    mascara = mascara.fillna(False)
    controles.append({"archivo": archivo, "control": control, "cantidad": int(mascara.sum()),
                      "ejemplos": df.loc[mascara].head(3).to_dict("records"), "decision": tratamiento})

for nombre, df in tablas.items():
    for col in [c for c in ("movieId", "userId", "tmdbId") if c in df.columns]:
        registrar(nombre, f"{col} no positivo", df[col].notna() & df[col].le(0), "Revisar origen; no corregir automáticamente")
registrar("movies.csv", "movieId repetido (todas las filas involucradas)", movies.duplicated("movieId", keep=False), "Revisar conflictos antes de depurar")
registrar("links.csv", "movieId repetido (todas las filas involucradas)", links.duplicated("movieId", keep=False), "Revisar relación uno a uno")
registrar("ratings.csv", "Par userId-movieId repetido", ratings.duplicated(["userId", "movieId"], keep=False), "Revisar eventos y fechas")
registrar("tags.csv", "Tripleta userId-movieId-tag repetida", tags.duplicated(["userId", "movieId", "tag"], keep=False), "Distinguir eventos de duplicados exactos")
valida = ratings["rating"].between(0.5, 5) & np.isclose((ratings["rating"] * 2).to_numpy(dtype=float, na_value=np.nan) % 1, 0)
registrar("ratings.csv", "Valoración nula o fuera de escala/paso", ~valida.fillna(False), "Investigar; no imputar con cero")
registrar("movies.csv", "Género no informado", movies["genres"].eq("(no genres listed)"), "Mantener indicador; decidir elegibilidad para clasificación")
registrar("links.csv", "tmdbId faltante", links["tmdbId"].isna(), "Conservar película; no inventar identificador externo")
registrar("links.csv", "imdbId ausente o no compuesto de dígitos", ~links["imdbId"].str.fullmatch(r"[0-9]+").fillna(False), "Revisar referencia externa")

fechas = {}
periodos = []
for nombre in ("ratings.csv", "tags.csv"):
    fechas[nombre] = pd.to_datetime(tablas[nombre]["timestamp"], unit="s", utc=True, errors="coerce")
    registrar(nombre, "Timestamp no interpretable", fechas[nombre].isna(), "Revisar fecha; conservar timestamp original")
    registrar(nombre, "Timestamp negativo", tablas[nombre]["timestamp"].lt(0), "Revisar plausibilidad para MovieLens")
    periodos.append({"archivo": nombre, "inicio_UTC": fechas[nombre].min(), "fin_UTC": fechas[nombre].max()})
reporte_calidad = pd.DataFrame(controles)
mostrar_tabla('Resultados de los controles de calidad', reporte_calidad[["archivo", "control", "cantidad", "decision"]])

#### Resultados de los controles de calidad

,archivo,control,cantidad,decision
0,movies.csv,movieId no positivo,0,Revisar origen; no corregir automáticamente
1,ratings.csv,movieId no positivo,0,Revisar origen; no corregir automáticamente
2,ratings.csv,userId no positivo,0,Revisar origen; no corregir automáticamente
3,tags.csv,movieId no positivo,0,Revisar origen; no corregir automáticamente
4,tags.csv,userId no positivo,0,Revisar origen; no corregir automáticamente
5,links.csv,movieId no positivo,0,Revisar origen; no corregir automáticamente
6,links.csv,tmdbId no positivo,0,Revisar origen; no corregir automáticamente
7,movies.csv,movieId repetido (todas las filas involucradas),0,Revisar conflictos antes de depurar
8,links.csv,movieId repetido (todas las filas involucradas),0,Revisar relación uno a uno
9,ratings.csv,Par userId-movieId repetido,0,Revisar eventos y fechas


### Interpretación: Validación de claves, valoraciones y fechas

No hay identificadores presentes no positivos ni movieId repetidos en movies o links. No se detectan pares usuario–película repetidos en ratings ni tripletas usuario–película–tag repetidas en tags bajo las reglas aplicadas. Todas las valoraciones respetan escala y paso de media estrella. Los timestamps son interpretables y no negativos; esto verifica su validez técnica, no la exactitud del evento original.

Los identificadores IMDb están presentes y compuestos por dígitos, pero no se verificó su existencia en el servicio externo. Las incidencias son 34 películas sin género informado y ocho referencias TMDB ausentes: la primera afecta objetivos de clasificación; la segunda, enlaces externos. Son problemas distintos y no justifican descartar automáticamente sus interacciones.

### Ejemplos de problemas de calidad

In [10]:
print("Ejemplos de incidencias:")
mostrar_tabla('Ejemplos de incidencias detectadas', reporte_calidad.loc[reporte_calidad["cantidad"].gt(0), ["archivo", "control", "ejemplos"]])

Ejemplos de incidencias:


#### Ejemplos de incidencias detectadas

,archivo,control,ejemplos
12,movies.csv,Género no informado,"[{'movieId': 114335, 'title': 'La cravate (1957)', 'genres': '(no genres listed)'}, {'..."
13,links.csv,tmdbId faltante,"[{'movieId': 791, 'imdbId': '0113610', 'tmdbId': None}, {'movieId': 1107, 'imdbId': '0..."


### Interpretación: Alcance de los ejemplos de incidencias

Se muestran hasta tres registros por control con incidencias, no todos los afectados. Los totales están en la columna cantidad del reporte anterior. Los ejemplos distinguen un marcador de ausencia semántica, `(no genres listed)`, de un nulo real en tmdbId. Deben orientar decisiones de elegibilidad para clasificación y conservación de enlaces faltantes. Los 15 títulos con espacios redundantes se reportan en el análisis de cadenas, no en esta tabla de controles.

### Período de las interacciones y escala observada

In [11]:
mostrar_tabla('Períodos observados en UTC', pd.DataFrame(periodos))

#### Períodos observados en UTC

,archivo,inicio_UTC,fin_UTC
0,ratings.csv,1996-03-29 18:36:55+00:00,2018-09-24 14:27:30+00:00
1,tags.csv,2006-01-13 19:09:12+00:00,2018-09-16 11:50:03+00:00


### Interpretación: Diferencias entre los períodos observados

Ratings abarca del **29 de marzo de 1996 al 24 de septiembre de 2018**, y tags del **13 de enero de 2006 al 16 de septiembre de 2018**. El período general del README no implica que ambas fuentes tengan el mismo intervalo. Las fechas se interpretan en UTC y los timestamps originales se conservan. Para una evaluación temporal, no se puede usar una etiqueta asignada después del corte como si ya estuviera disponible para una valoración anterior.

In [12]:
print("Escala observada:", sorted(ratings["rating"].dropna().unique().tolist()))

Escala observada: [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]


### Interpretación: Escala de puntuaciones

Aparecen los diez niveles posibles de 0,5 a 5,0. Cada nivel se observa al menos una vez, pero esta lista no informa su frecuencia ni implica equilibrio. El EDA estudiará la distribución y la concentración de puntuaciones. El criterio de valoración favorable debe definirse expresamente antes de construir perfiles.

### Interpretación de calidad
No hay duplicados exactos en ninguno de los cuatro archivos. No se detectan valoraciones fuera de escala, identificadores internos no positivos ni timestamps no interpretables. Los intervalos observados se muestran en UTC; la conversión se mantiene aparte de las tablas originales.

Los ocho `tmdbId` faltantes afectan referencias externas, no la existencia de la película ni sus interacciones. Las 34 películas con `(no genres listed)` no tienen un nulo de CSV, pero sí carecen de géneros informados: es una ausencia semántica que importa para la futura clasificación. Ambas incidencias deben documentarse sin eliminar películas durante la inspección.

# 7. Relaciones y cobertura inicial

- `movies.movieId` → `ratings.movieId`: una película puede tener muchas valoraciones.
- `movies.movieId` → `tags.movieId`: una película puede tener muchos eventos de etiquetas.
- `movies.movieId` ↔ `links.movieId`: se comprueba una relación uno a uno.
- `userId` identifica al mismo usuario entre ratings y tags; ambas tablas contienen eventos y no deben unirse directamente por usuario o película para calcular conteos.

Se agregan las interacciones antes de unirlas al catálogo. La cobertura textual inicial exige al menos una etiqueta no nula y no vacía después de retirar espacios externos; no representa todavía la cobertura posterior a la normalización.

In [13]:
catalogo_ids = set(movies["movieId"].dropna())
integridad = pd.DataFrame([
    {"archivo": nombre, "filas_movieId_fuera_catalogo": int((~df["movieId"].isin(catalogo_ids)).sum()),
     "peliculas_fuera_catalogo": int(df.loc[~df["movieId"].isin(catalogo_ids), "movieId"].nunique())}
    for nombre, df in tablas.items() if nombre != "movies.csv"
])
mostrar_tabla('Integridad referencial respecto del catálogo de películas', integridad)

#### Integridad referencial respecto del catálogo de películas

,archivo,filas_movieId_fuera_catalogo,peliculas_fuera_catalogo
0,ratings.csv,0,0
1,tags.csv,0,0
2,links.csv,0,0


### Interpretación: Integridad referencial

Los ceros indican que ningún evento ni enlace referencia una película ausente del catálogo. Filas fuera del catálogo cuenta eventos afectados; películas fuera del catálogo cuenta identificadores distintos afectados. Una referencia inválida podría afectar muchas filas, por lo que son medidas diferentes. Esto asegura correspondencia hacia movies, pero no prueba que todas las películas tengan eventos o enlaces: la cobertura inversa se verifica a continuación.

### Agregación segura y cálculo de cobertura

In [14]:
print("Películas sin fila en links:", len(catalogo_ids - set(links["movieId"].dropna())))
print("Usuarios que etiquetan y no aparecen en ratings:", len(set(tags["userId"]) - set(ratings["userId"])))

etiqueta_utilizable = tags["tag"].notna() & tags["tag"].str.strip().ne("").fillna(False)
conteo_ratings = ratings.groupby("movieId").size().rename("n_valoraciones")
conteo_tags = tags.groupby("movieId").size().rename("n_etiquetas")
conteo_texto = tags.loc[etiqueta_utilizable].groupby("movieId").size().rename("n_etiquetas_no_vacias")
catalogo_inspeccion = (movies.merge(links, on="movieId", how="left", validate="one_to_one")
    .merge(conteo_ratings, on="movieId", how="left", validate="one_to_one")
    .merge(conteo_tags, on="movieId", how="left", validate="one_to_one")
    .merge(conteo_texto, on="movieId", how="left", validate="one_to_one"))
columnas_conteo = ["n_valoraciones", "n_etiquetas", "n_etiquetas_no_vacias"]
catalogo_inspeccion[columnas_conteo] = catalogo_inspeccion[columnas_conteo].fillna(0).astype("int64")
assert len(catalogo_inspeccion) == len(movies)
assert catalogo_inspeccion["n_valoraciones"].sum() == len(ratings)
assert catalogo_inspeccion["n_etiquetas"].sum() == len(tags)

r = catalogo_inspeccion["n_valoraciones"].gt(0)
t = catalogo_inspeccion["n_etiquetas"].gt(0)
texto = catalogo_inspeccion["n_etiquetas_no_vacias"].gt(0)
grupos = {"Con valoraciones": r, "Con etiquetas": t, "Con texto inicial no vacío": texto,
          "Con ambas interacciones": r & t, "Solo valoraciones": r & ~t,
          "Solo etiquetas": ~r & t, "Sin interacciones": ~r & ~t}
cobertura = pd.DataFrame([
    {"grupo": grupo, "peliculas": int(mask.sum()), "denominador_catalogo": len(movies),
     "porcentaje_catalogo": round(100 * mask.sum() / len(movies), 2)}
    for grupo, mask in grupos.items()
])
mostrar_tabla('Cobertura inicial sobre el catálogo completo', cobertura)

Películas sin fila en links: 0
Usuarios que etiquetan y no aparecen en ratings: 0


#### Cobertura inicial sobre el catálogo completo

,grupo,peliculas,denominador_catalogo,porcentaje_catalogo
0,Con valoraciones,9724,9742,99.82
1,Con etiquetas,1572,9742,16.14
2,Con texto inicial no vacío,1572,9742,16.14
3,Con ambas interacciones,1554,9742,15.95
4,Solo valoraciones,8170,9742,83.86
5,Solo etiquetas,18,9742,0.18
6,Sin interacciones,0,9742,0.00


### Interpretación: Cobertura y categorías que se solapan

No hay películas sin fila en links y todos los usuarios que etiquetan aparecen también en ratings. La cobertura usa 9 742 películas como denominador: 9 724 tienen valoraciones y 1 572 tienen etiquetas no vacías (**16,14 %**). Las filas con valoraciones, con etiquetas y con ambas se solapan y no deben sumarse.

Las categorías exclusivas sí dividen el catálogo: 8 170 solo con valoraciones, 18 solo con etiquetas, 1 554 con ambas y cero sin interacciones. Igualar cobertura con etiquetas y con texto inicial no vacío indica que el filtro inicial no elimina películas; la normalización podría cambiarlo. Agregar eventos antes de unir y comprobar las sumas evita multiplicarlos al combinar ratings y tags.

### Cobertura de usuarios y películas por archivo

In [15]:
unicos = pd.DataFrame([
    {"archivo": nombre, "peliculas_unicas": df["movieId"].nunique(),
     "usuarios_unicos": df["userId"].nunique() if "userId" in df else pd.NA}
    for nombre, df in tablas.items()
])
mostrar_tabla('Películas y usuarios únicos por archivo', unicos)

#### Películas y usuarios únicos por archivo

,archivo,peliculas_unicas,usuarios_unicos
0,movies.csv,9742,<NA>
1,ratings.csv,9724,610
2,tags.csv,1572,58
3,links.csv,9742,<NA>


### Interpretación: Usuarios y películas únicos

Las 100 836 valoraciones provienen de 610 usuarios y corresponden a 9 724 películas. Las 3 683 etiquetas provienen de 58 usuarios y describen 1 572 películas. Por tanto, cantidad de eventos y cantidad de entidades únicas no son intercambiables. En movies y links, usuarios únicos no aplica porque no existe userId; el valor ausente de ese resumen no representa un problema de usuarios sin identificar.

### Inspección del catálogo relacionado

In [16]:
print(f"Usuarios que etiquetan respecto de usuarios que valoran: {tags.userId.nunique()} / {ratings.userId.nunique()} = {100 * tags.userId.nunique() / ratings.userId.nunique():.2f}%")
mostrar_tabla('Muestra del catálogo con enlaces y conteos de interacciones', catalogo_inspeccion.head())

Usuarios que etiquetan respecto de usuarios que valoran: 58 / 610 = 9.51%


#### Muestra del catálogo con enlaces y conteos de interacciones

,movieId,title,genres,imdbId,tmdbId,n_valoraciones,n_etiquetas,n_etiquetas_no_vacias
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,0114709,862,215,3,3
1,2,Jumanji (1995),Adventure|Children|Fantasy,0113497,8844,110,4,4
2,3,Grumpier Old Men (1995),Comedy|Romance,0113228,15602,52,2,2
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,0114885,31357,7,0,0
4,5,Father of the Bride Part II (1995),Comedy,0113041,11862,49,2,2


### Interpretación: Catálogo agregado y participación textual

El catálogo relacionado mantiene una fila por película. Sus conteos indican número de valoraciones y etiquetas asociadas; un cero expresa ausencia de eventos, no una valoración de cero. Los 58 usuarios que etiquetan representan el **9,51 %** de quienes valoran, mientras que el **16,14 %** mide cobertura por película. Son proporciones distintas: quién aporta texto y cuántas películas lo tienen. Ninguna demuestra por sí misma representatividad de los gustos de todo el grupo.

### Interpretación de las relaciones
Todas las referencias `movieId` de ratings, tags y links pertenecen al catálogo y cada película dispone de una fila en links. La agregación conserva las cantidades originales de eventos, evitando multiplicar valoraciones al relacionarlas con etiquetas.

De las 9 742 películas, 9 724 tienen valoraciones y 1 572 tienen etiquetas originales no vacías: la cobertura textual inicial es **16,14 %**. Hay 1 554 películas con ambos tipos de interacción, 8 170 solo con valoraciones y 18 solo con etiquetas; ninguna está sin interacciones.

Participan 610 usuarios en valoraciones y 58 en etiquetas. Los usuarios que etiquetan representan el **9,51 %** de quienes valoran. Estas coberturas muestran que el catálogo textual es un subconjunto y que su contenido procede de una minoría de usuarios; no permiten inferir representatividad de preferencias.

# 8. Conclusiones y transición al EDA

La celda final resume indicadores observados y comprueba que no cambió el contenido de los CSV. Los indicadores corresponden a los archivos locales, no a resultados de modelos.

In [17]:
indicadores = pd.DataFrame([
    ("Películas del catálogo", len(movies)),
    ("Filas de enlaces externos", len(links)),
    ("Valoraciones", len(ratings)),
    ("Asignaciones de etiquetas", len(tags)),
    ("Usuarios con valoraciones", ratings.userId.nunique()),
    ("Usuarios con etiquetas", tags.userId.nunique()),
    ("Películas con texto inicial no vacío", int(texto.sum())),
    ("Cobertura textual inicial (%)", round(100 * texto.sum() / len(movies), 2)),
    ("Películas sin género informado", int(movies.genres.eq("(no genres listed)").sum())),
    ("Enlaces TMDB faltantes", int(links.tmdbId.isna().sum())),
], columns=["indicador", "valor"])
mostrar_tabla('Indicadores finales de la inspección', indicadores)

#### Indicadores finales de la inspección

,indicador,valor
0,Películas del catálogo,9742.00
1,Filas de enlaces externos,9742.00
2,Valoraciones,100836.00
3,Asignaciones de etiquetas,3683.00
4,Usuarios con valoraciones,610.00
5,Usuarios con etiquetas,58.00
6,Películas con texto inicial no vacío,1572.00
7,Cobertura textual inicial (%),16.14
8,Películas sin género informado,34.00
9,Enlaces TMDB faltantes,8.00


### Interpretación: Síntesis y límites de los indicadores

El resumen distingue películas, enlaces, eventos y usuarios. La correspondencia completa entre movies y links se respalda por conteos, unicidad y referencias, no únicamente por igualdad de filas. El catálogo textual inicial comprende 1 572 películas frente a 9 742 totales.

La clasificación requiere además géneros informados; todavía no se calculó cuántas películas cumplen simultáneamente ambos requisitos. No debe restarse automáticamente 34 o 8 del catálogo textual: son incidencias del catálogo completo y no se estableció su intersección con las películas etiquetadas. Estos indicadores orientan el EDA y la preparación, pero no son métricas de modelos.

### Comprobación de conservación de los archivos originales

In [18]:
assert {nombre: huella(RAW / nombre) for nombre in ARCHIVOS} == huellas_iniciales
print("Verificación final: los cuatro CSV originales permanecen intactos.")

Verificación final: los cuatro CSV originales permanecen intactos.


### Interpretación: Conservación de los datos originales

Las cuatro huellas finales coinciden con las iniciales, por lo que el contenido de los CSV permaneció intacto durante la ejecución. Las conversiones de tipos, fechas y agregaciones se realizaron en memoria. Esta comprobación permite partir de los mismos originales en los notebooks siguientes y documentar allí las transformaciones.

## Conclusiones

1. Los cuatro archivos se cargan correctamente y sus relaciones con el catálogo son consistentes. El diccionario incluye las 14 columnas y preserva la naturaleza de sus identificadores.
2. La inspección identifica ocho referencias TMDB ausentes y 34 películas sin género informado. Estas situaciones requieren decisiones distintas en el preprocesamiento.
3. El texto disponible corresponde a etiquetas libres, sin sinopsis. Su cobertura inicial del 16,14 % limita la extensión de la propuesta basada exclusivamente en contenido textual.
4. La disponibilidad de valoraciones es mucho mayor que la de etiquetas. Debe distinguirse el catálogo completo del subconjunto textual y documentarse cómo se tratarán películas sin texto.

## Preguntas para el notebook 02

- ¿Qué géneros predominan y cuáles tienen menor cobertura de etiquetas?
- ¿Cómo se distribuyen las puntuaciones y la actividad entre usuarios y películas?
- ¿Qué etiquetas o frases son frecuentes y qué variantes requieren normalización?
- ¿Cómo cambia la disponibilidad de texto según género y volumen de valoraciones?

## Decisiones pendientes para el notebook 03

Conservar los identificadores externos faltantes sin inventarlos; definir la elegibilidad de películas sin géneros para clasificación; normalizar etiquetas con trazabilidad y recalcular la cobertura de texto utilizable. Mantener eventos y timestamps para diseñar una futura evaluación temporal sin información posterior al corte.

## Revisión

- [x] Carga e inspección de los cuatro CSV.
- [x] Diccionario de variables completo.
- [x] Controles de calidad, ejemplos y períodos en UTC.
- [x] Integridad referencial y cobertura con denominadores explícitos.
- [x] Comprobación de conservación de los originales.